# 🌿 Plant Health Detector — Web Application Notebook

This notebook walks through the **Flask web application** (`app.py`) that serves the fine-tuned CLIP model as a browser-based plant disease detector.

**Architecture overview:**
```
Browser (HTML + JS)
    │  POST /predict  (image + threshold)
    ▼
Flask Server
    │
    ├── Fine-tuned CLIP  ──→  Top-5 class probabilities
    │       (if conf >= threshold)
    │
    └── Zero-shot CLIP   ──→  Semantic description match
            (if conf < threshold)         
    │
    ▼
JSON response → Browser renders result
```

**Files needed before running:**
- `clip_plant_best.pt` — fine-tuned weights from training notebook
- `clip_labels.json` — class list + prompts saved during training

## Section 1 — Imports

- `flask` — lightweight Python web framework to serve the app
- `io` / `base64` — for handling image bytes sent by the browser
- `clip`, `torch`, `PIL` — same model stack as training
- `json`, `os`, `numpy` — utilities

In [ ]:
import torch
import clip
import torch.nn as nn
from PIL import Image
import json, os, io, base64
import numpy as np
from flask import Flask, request, jsonify, render_template_string

print('All imports successful!')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

## Section 2 — Configuration

- `MODEL_PATH` — path to the saved fine-tuned weights
- `LABELS_PATH` — path to the JSON file with class names and prompts
- `FALLBACK_THRESHOLD` — if the fine-tuned model's top confidence is **below** this %, we switch to zero-shot CLIP for a more descriptive (less specific) answer
- `MAX_IMAGE_SIZE` — large images are resized before inference to keep latency low

In [ ]:
MODEL_PATH         = "clip_plant_best.pt"
LABELS_PATH        = "clip_labels.json"
DEVICE             = "cuda" if torch.cuda.is_available() else "cpu"
FALLBACK_THRESHOLD = 70.0   # % — adjustable in the UI slider too
MAX_IMAGE_SIZE     = 1024   # pixels — longer edge capped here

print(f'Device             : {DEVICE}')
print(f'Fallback threshold : {FALLBACK_THRESHOLD}%')
print(f'Max image size     : {MAX_IMAGE_SIZE}px')

## Section 3 — Disease Database

A hand-curated dictionary mapping each class label to:
- **cause** — the pathogen or pest responsible
- **severity** — `CRITICAL / HIGH / MEDIUM / LOW / NONE`
- **treatment** — actionable remediation steps

This lookup is done after prediction and included in the JSON response to the browser. The browser uses the `severity` field to colour the disease info box (red for CRITICAL, orange for HIGH, etc.).

If a predicted label is not in the database, a default fallback message is returned.

In [ ]:
DISEASE_DB = {
    "Tomato___Late_blight":         {"cause":"Phytophthora infestans",         "severity":"HIGH",     "treatment":"Apply copper-based fungicide immediately. Remove and destroy infected leaves. Avoid overhead watering."},
    "Tomato___Early_blight":        {"cause":"Alternaria solani",               "severity":"MEDIUM",   "treatment":"Apply chlorothalonil or mancozeb fungicide. Remove lower infected leaves. Stake plants for airflow."},
    "Tomato___Bacterial_spot":      {"cause":"Xanthomonas spp.",                "severity":"MEDIUM",   "treatment":"Spray copper bactericide. Avoid working with wet plants. Use disease-free seeds."},
    "Tomato___Leaf_Mold":           {"cause":"Passalora fulva",                 "severity":"MEDIUM",   "treatment":"Reduce humidity below 85%. Apply fungicide. Prune lower leaves."},
    "Tomato___Septoria_leaf_spot":  {"cause":"Septoria lycopersici",            "severity":"MEDIUM",   "treatment":"Remove infected leaves. Apply mancozeb every 7-10 days."},
    "Tomato___Spider_mites Two-spotted_spider_mite":{"cause":"Tetranychus urticae","severity":"MEDIUM","treatment":"Spray neem oil or insecticidal soap. Increase humidity."},
    "Tomato___Target_Spot":         {"cause":"Corynespora cassiicola",          "severity":"MEDIUM",   "treatment":"Apply azoxystrobin fungicide. Remove infected leaves."},
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus":{"cause":"TYLCV (whitefly-transmitted)","severity":"HIGH","treatment":"No cure. Remove infected plants. Control whiteflies with insecticide."},
    "Tomato___Tomato_mosaic_virus": {"cause":"ToMV virus",                      "severity":"HIGH",     "treatment":"No cure. Remove and destroy plant. Disinfect all tools with bleach."},
    "Tomato___healthy":             {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Potato___Early_blight":        {"cause":"Alternaria solani",               "severity":"MEDIUM",   "treatment":"Apply chlorothalonil fungicide. Rotate crops every 3 years. Improve soil drainage."},
    "Potato___Late_blight":         {"cause":"Phytophthora infestans",          "severity":"HIGH",     "treatment":"Apply metalaxyl + mancozeb immediately. Destroy all infected plants."},
    "Potato___healthy":             {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Apple___Apple_scab":           {"cause":"Venturia inaequalis",             "severity":"MEDIUM",   "treatment":"Apply myclobutanil or captan fungicide in spring. Rake and destroy fallen leaves."},
    "Apple___Black_rot":            {"cause":"Botryosphaeria obtusa",           "severity":"HIGH",     "treatment":"Prune infected branches 15cm below infection. Apply captan fungicide."},
    "Apple___Cedar_apple_rust":     {"cause":"Gymnosporangium juniperi-virginianae","severity":"MEDIUM","treatment":"Apply myclobutanil in spring. Remove nearby juniper trees if possible."},
    "Apple___healthy":              {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot":{"cause":"Cercospora zeae-maydis","severity":"MEDIUM","treatment":"Apply foliar fungicide. Plant resistant hybrids."},
    "Corn_(maize)___Common_rust_":  {"cause":"Puccinia sorghi",                 "severity":"LOW",      "treatment":"Apply triazole fungicide if severe. Plant resistant varieties."},
    "Corn_(maize)___Northern_Leaf_Blight":{"cause":"Exserohilum turcicum",      "severity":"MEDIUM",   "treatment":"Apply propiconazole fungicide at tasseling. Crop rotation."},
    "Corn_(maize)___healthy":       {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Grape___Black_rot":            {"cause":"Guignardia bidwellii",            "severity":"HIGH",     "treatment":"Apply mancozeb or myclobutanil. Remove mummified berries."},
    "Grape___Esca_(Black_Measles)": {"cause":"Fungal complex (Phaeomoniella)",  "severity":"HIGH",     "treatment":"No effective cure. Prune infected wood. Apply wound protectant."},
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)":{"cause":"Isariopsis clavispora","severity":"MEDIUM","treatment":"Apply copper-based fungicide. Improve canopy airflow."},
    "Grape___healthy":              {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Peach___Bacterial_spot":       {"cause":"Xanthomonas arboricola",          "severity":"MEDIUM",   "treatment":"Apply copper bactericide in spring. Avoid overhead irrigation."},
    "Peach___healthy":              {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Pepper,_bell___Bacterial_spot":{"cause":"Xanthomonas campestris",          "severity":"MEDIUM",   "treatment":"Spray copper bactericide. Use disease-free seeds."},
    "Pepper,_bell___healthy":       {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Squash___Powdery_mildew":      {"cause":"Podosphaera xanthii",             "severity":"LOW",      "treatment":"Spray potassium bicarbonate or sulfur. Improve air circulation."},
    "Strawberry___Leaf_scorch":     {"cause":"Diplocarpon earlianum",           "severity":"MEDIUM",   "treatment":"Apply captan fungicide. Remove old leaves. Improve drainage."},
    "Strawberry___healthy":         {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Cherry_(including_sour)___Powdery_mildew":{"cause":"Podosphaera clandestina","severity":"LOW",   "treatment":"Apply sulfur spray. Prune for air circulation."},
    "Cherry_(including_sour)___healthy":{"cause":"None",                        "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Orange___Haunglongbing_(Citrus_greening)":{"cause":"Candidatus Liberibacter","severity":"CRITICAL","treatment":"No cure. Remove and destroy infected tree immediately. Control Asian citrus psyllid."},
    "Raspberry___healthy":          {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Soybean___healthy":            {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
    "Blueberry___healthy":          {"cause":"None",                            "severity":"NONE",     "treatment":"Plant is healthy. Continue regular monitoring."},
}

print(f'Disease database loaded: {len(DISEASE_DB)} entries')

## Section 4 — Zero-Shot Fallback Prompts

When the fine-tuned model is not confident (below threshold), we switch to **zero-shot CLIP** using a set of 30 hand-written natural language descriptions. These describe general disease patterns and visual symptoms.

**Why zero-shot as fallback?**
- The fine-tuned model knows exactly 38 classes. An out-of-distribution image (wrong plant species, blurry, partial leaf) will still be forced into one of those 38 classes.
- Zero-shot CLIP can return a more semantically accurate description like "a leaf with powdery mildew" without committing to a specific class.
- This is especially useful for plant species NOT in the training set.

In [ ]:
ZEROSHOT_PROMPTS = [
    "a healthy green plant leaf with no spots, no damage, completely normal",
    "a plant leaf with early blight disease showing brown concentric ring spots",
    "a plant leaf with late blight disease showing dark water-soaked lesions",
    "a tomato plant leaf infected with bacterial spot disease",
    "a potato leaf showing signs of early blight with Alternaria fungal infection",
    "a potato plant leaf infected with late blight Phytophthora disease",
    "a tomato leaf showing yellow leaf curl virus symptoms",
    "a grape leaf with black rot fungal disease spots",
    "a leaf covered with white powdery mildew coating on the surface",
    "a corn leaf with gray leaf spot or cercospora disease lesions",
    "a corn plant leaf with orange rust pustules from common rust disease",
    "a corn leaf with northern leaf blight showing elongated gray lesions",
    "an apple leaf with olive-green scab lesions from apple scab disease",
    "an apple leaf showing black rot disease symptoms",
    "a leaf showing signs of spider mite infestation with stippling damage",
    "a tomato leaf showing mosaic virus symptoms with mottled coloring",
    "a strawberry leaf with scorch disease showing purple spots",
    "a pepper leaf with bacterial spot disease",
    "a peach leaf showing bacterial spot disease symptoms",
    "a citrus leaf showing Huanglongbing greening disease",
    "a plant leaf showing septoria leaf spot disease",
    "a plant leaf showing target spot disease with concentric rings",
    "a plant leaf showing leaf mold disease with olive-green patches",
    "a grape leaf showing Esca black measles disease",
    "a grape leaf with isariopsis leaf blight symptoms",
    "an entire plant showing multiple diseased leaves from above",
    "multiple plant leaves photographed together showing disease",
    "a close-up photo of a diseased leaf with unclear focus",
    "a plant stem and leaves showing disease symptoms",
    "an overhead view of a plant with unhealthy yellowing leaves",
]

print(f'Zero-shot prompts: {len(ZEROSHOT_PROMPTS)} descriptions loaded')

## Section 5 — Load Model & Precompute Embeddings

At startup we:
1. Load CLIP ViT-B/16 and the fine-tuned weights
2. **Precompute** text embeddings for both the 38 fine-tuned class prompts AND the 30 zero-shot prompts

**Why precompute?**  
Text encoding is slow (~100ms). Doing it once at startup and caching means each image request only needs a single `encode_image` call (~20ms on GPU). This makes the web app feel instant.

The text features are stored as tensors in memory and shared across all requests.

In [ ]:
print(f'Loading CLIP on {DEVICE}...')
model, preprocess = clip.load("ViT-B/16", device=DEVICE)
model = model.float()
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print('Fine-tuned model loaded ✓')

# Load class list from training
data    = json.load(open(LABELS_PATH))
CLASSES = data["classes"]
PROMPTS = data["prompts"]

# Precompute fine-tuned text embeddings (38 class prompts)
with torch.no_grad():
    text_tokens_ft   = clip.tokenize(PROMPTS, truncate=True).to(DEVICE)
    TEXT_FEATURES_FT = model.encode_text(text_tokens_ft)
    TEXT_FEATURES_FT = nn.functional.normalize(TEXT_FEATURES_FT, dim=-1)

# Precompute zero-shot text embeddings (30 descriptive prompts)
with torch.no_grad():
    text_tokens_zs   = clip.tokenize(ZEROSHOT_PROMPTS, truncate=True).to(DEVICE)
    TEXT_FEATURES_ZS = model.encode_text(text_tokens_zs)
    TEXT_FEATURES_ZS = TEXT_FEATURES_ZS / TEXT_FEATURES_ZS.norm(dim=-1, keepdim=True)

print(f'Fine-tuned text embeddings : {TEXT_FEATURES_FT.shape}  ({len(CLASSES)} classes)')
print(f'Zero-shot text embeddings  : {TEXT_FEATURES_ZS.shape}  ({len(ZEROSHOT_PROMPTS)} prompts)')
print(f'Fallback threshold         : {FALLBACK_THRESHOLD}%')

## Section 6 — Inference Engine

Three functions handle the prediction pipeline:

### `run_finetuned(image_tensor)`
Encodes the image, computes cosine similarity with fine-tuned class embeddings, returns softmax probabilities.

### `run_zeroshot(image_tensor)`
Same process but against the 30 zero-shot prompts. Returns the closest semantic description.

### `predict(pil_image, threshold)`
The main entry point:
1. Preprocess the PIL image using CLIP's transform
2. Run fine-tuned model
3. If top confidence < threshold → also run zero-shot
4. Build and return a rich JSON dict with all results + disease database info

In [ ]:
def run_finetuned(image_tensor):
    """Run fine-tuned CLIP → returns (softmax probabilities, class list)."""
    with torch.no_grad():
        img_feat = model.encode_image(image_tensor)
        img_feat = nn.functional.normalize(img_feat, dim=-1)
        logits   = 100.0 * (img_feat @ TEXT_FEATURES_FT.T)
        probs    = logits.softmax(dim=-1)[0].cpu().numpy()
    return probs, CLASSES


def run_zeroshot(image_tensor):
    """Run zero-shot CLIP → returns (softmax probabilities, prompt list)."""
    with torch.no_grad():
        img_feat  = model.encode_image(image_tensor)
        txt_feat  = TEXT_FEATURES_ZS.clone()
        img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
        similarity = (100.0 * img_feat @ txt_feat.T).softmax(dim=-1)
        probs      = similarity[0].cpu().numpy()
    return probs, ZEROSHOT_PROMPTS


def predict(pil_image: Image.Image, threshold: float = FALLBACK_THRESHOLD) -> dict:
    """
    Full prediction pipeline.
    Returns a dict ready to be serialized as JSON and sent to the browser.
    """
    # Preprocess: resize + normalize to CLIP's expected format (224x224, normalized)
    img_tensor = preprocess(pil_image).unsqueeze(0).to(DEVICE)

    # Step 1: Fine-tuned model
    ft_probs, ft_classes = run_finetuned(img_tensor)
    top5_idx  = ft_probs.argsort()[::-1][:5]
    top_conf  = float(ft_probs[top5_idx[0]]) * 100
    top_label = ft_classes[top5_idx[0]]

    parts      = top_label.split("___")
    plant      = parts[0].replace("_", " ")
    condition  = parts[1].replace("_", " ") if len(parts) > 1 else "Unknown"
    is_healthy = "healthy" in top_label.lower()

    top5 = [
        {"label": ft_classes[i].replace("___", " — ").replace("_", " "),
         "confidence": round(float(ft_probs[i]) * 100, 2)}
        for i in top5_idx
    ]

    db_info = DISEASE_DB.get(top_label, {})
    used_fallback = False

    # Step 2: Fallback if confidence too low
    zeroshot_results = []
    if top_conf < threshold:
        used_fallback = True
        zs_probs, zs_prompts = run_zeroshot(img_tensor)
        top3_zs = zs_probs.argsort()[::-1][:3]
        zeroshot_results = [
            {"description": zs_prompts[i], "confidence": round(float(zs_probs[i]) * 100, 2)}
            for i in top3_zs
        ]

    return {
        "plant":            plant,
        "condition":        condition,
        "is_healthy":       is_healthy,
        "confidence":       round(top_conf, 2),
        "used_fallback":    used_fallback,
        "threshold_used":   threshold,
        "top5":             top5,
        "zeroshot_results": zeroshot_results,
        "disease_info": {
            "cause":     db_info.get("cause",     "Consult agricultural specialist"),
            "severity":  db_info.get("severity",  "MEDIUM"),
            "treatment": db_info.get("treatment", "Consult local agricultural extension office"),
        }
    }


# Quick smoke test (replace with an actual image path to test)
print('predict() function defined ✓')
print('Usage: result = predict(Image.open("leaf.jpg"), threshold=70.0)')
print('Returns:', list(predict.__annotations__.values())[-1])

## Section 7 — HTML Template (Frontend)

The entire frontend is a single HTML string embedded in Python (using Flask's `render_template_string`).

**Key UI components:**

| Component | Description |
|-----------|-------------|
| Drop zone | Accepts click, drag & drop, and clipboard paste (Ctrl+V) |
| Threshold slider | Real-time control for the fallback threshold (0–100%) |
| Preview panel | Shows the selected image before analysis |
| Result panel | Renders badges, confidence bar, disease info, top-5 chart |
| Fallback section | Only visible when zero-shot mode triggers |

**CSS design:**
- Dark green theme using CSS custom properties (`--green-dark`, `--green-lite`, etc.)
- Responsive: single column on mobile, two-column grid on desktop
- Severity colours: CRITICAL=red, HIGH=orange, MEDIUM=amber, LOW/NONE=green

**JavaScript flow:**
```
User selects file / drops / pastes
    ↓
FileReader.readAsDataURL()  →  show preview
    ↓
analyse() called
    ↓
FormData(image + threshold)  →  POST /predict
    ↓
renderResult(data)  →  build HTML and inject into #result-panel
```

> The HTML is too long to show in full here — it's defined as the `HTML` constant in `app.py`. Below we just note its structure.

In [ ]:
# The HTML template is stored as a multi-line string in app.py.
# Here we print its structural summary for reference.

html_structure = """
HTML Structure
═══════════════════════════════════════════
<head>
  <style>  ← All CSS (dark green theme, grid layout, badges)

<body>
  <header>          ← Logo + title + threshold slider
  <div.layout>      ← CSS Grid: 2 columns
    <div.card>      ← LEFT: Upload panel
      drop-zone     ← Click / drag / paste
      file-input    ← Hidden <input type=file>
      preview-wrap  ← Image preview + filename + size
      analyse-btn   ← Triggers AJAX request
    <div.card>      ← RIGHT: Result panel
      empty-state   ← Shown before analysis
      spinner-wrap  ← Shown during request
      result-panel  ← Populated by renderResult()

<script>
  Threshold slider event listener
  Drag & drop events (dragover, dragleave, drop)
  Clipboard paste handler
  loadFile(file)   → FileReader + show preview
  clearImage()     → Reset UI state
  analyse()        → POST /predict → renderResult()
  renderResult(d)  → Build result HTML from JSON response
"""
print(html_structure)

## Section 8 — Flask Routes

The Flask app exposes two routes:

### `GET /`
Renders the HTML page. The current `FALLBACK_THRESHOLD` is passed as a template variable so the slider starts at the correct position.

### `POST /predict`
Accepts `multipart/form-data` with:
- `image` — the uploaded image file
- `threshold` — the current slider value from the UI

Steps:
1. Read file bytes
2. Open as PIL image, convert to RGB
3. Resize if larger than `MAX_IMAGE_SIZE`
4. Call `predict()` → returns dict
5. Return `jsonify(result)` → browser receives JSON

Error handling: any exception returns `{"error": "..."}` with HTTP 500.

In [ ]:
# ── Paste the full HTML string here (truncated for notebook clarity) ──
# In production this is the HTML variable from app.py
HTML = """<!DOCTYPE html><html><head><title>Plant Health Detector</title></head>
<body><h1>Plant Health Detector</h1>
<p>Full HTML template is in app.py — run that file to launch the web server.</p>
</body></html>"""

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 32 * 1024 * 1024   # 32 MB max upload

@app.route('/')
def index():
    """Serve the main HTML page."""
    return render_template_string(HTML, threshold=int(FALLBACK_THRESHOLD))


@app.route('/predict', methods=['POST'])
def predict_route():
    """Accept image upload → run inference → return JSON result."""
    if 'image' not in request.files:
        return jsonify({"error": "No image in request"}), 400

    file      = request.files['image']
    threshold = float(request.form.get('threshold', FALLBACK_THRESHOLD))

    try:
        img_bytes = file.read()
        pil_img   = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        # Resize very large images to keep inference fast
        w, h = pil_img.size
        if max(w, h) > MAX_IMAGE_SIZE:
            ratio   = MAX_IMAGE_SIZE / max(w, h)
            pil_img = pil_img.resize((int(w*ratio), int(h*ratio)), Image.LANCZOS)

        result = predict(pil_img, threshold=threshold)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)}), 500


print('Flask routes defined:')
print('  GET  /         → serve HTML page')
print('  POST /predict  → run inference, return JSON')

## Section 9 — Launch the Server

Running the cell below starts the Flask server.

- **Local access:** http://localhost:5000
- **Network access:** http://\<your-ip\>:5000  (accessible to other devices on the same LAN)

> **In Jupyter**: The server blocks the kernel while running. Open the URL in a browser tab.  
> **To stop**: Press the ■ Stop button in the toolbar (sends KeyboardInterrupt).

**Production tip:** For public deployment, replace `debug=False` with a proper WSGI server like `gunicorn`:
```bash
gunicorn -w 1 -b 0.0.0.0:5000 app:app
```
(Use 1 worker to avoid loading the model multiple times)

In [ ]:
import socket
try:
    local_ip = socket.gethostbyname(socket.gethostname())
except Exception:
    local_ip = "127.0.0.1"

print('=' * 55)
print('  🌿 Plant Health Detector — Starting...')
print('=' * 55)
print(f'  Local  : http://localhost:5000')
print(f'  Network: http://{local_ip}:5000')
print(f'  Threshold : {FALLBACK_THRESHOLD}% (adjustable in UI)')
print('  Press Ctrl+C or Stop button to quit')
print('=' * 55)

app.run(host='0.0.0.0', port=5000, debug=False)